# Validation empirique de la greffe gatée — Expériences E1 et E2

Notebook **indépendant**, conversion réelle (pas un simple habillage) du script
`experiments_surgery.jl` fourni : les 8 fonctions ADAPTER sont reliées à la vraie
API NeuroDSL, ce qui a nécessité de construire un nouveau mécanisme —
`graft_shadow_block!` (`src/graph_surgery.jl`) — absent avant cette session.

- **E1 : bit-exactness de la greffe** — comparaison `reinterpret(UInt32, logits)`
  avant/après greffe, avec vérification explicite des préconditions IEEE-754.
- **E2 : plasticité vs point de selle** — trois bras avec master seed partagé,
  aucun filtrage post-hoc :
  - **rezero** : `alpha=0`, `theta` aléatoire — *Gradient Shadowing / ReZero*.
  - **net2net** : `alpha=1`, projections de sortie à zéro — *Fixup/Net2Net*.
  - **degenerate** : `alpha=0` ET projections de sortie à zéro — doit rester gelé.

**Écart avec le script d'origine** (documenté, pas caché) : NeuroDSL n'a pas de
dimension batch explicite sur `Embedding`/`LlamaModel` (un `:token_ids` est un
`Vector{Int}`, une séquence à la fois — confirmé dans `src/synthetic_circuits.jl`).
`cfg.batch` est donc abandonné ; chaque « pas » entraîne sur une séquence, pas un
lot. Ça ne change aucune des deux questions posées (bit-exactness et signe du
gradient sont des propriétés par séquence, pas des statistiques de lot).

In [1]:
using NeuroDSL, Random, Printf, Statistics, DelimitedFiles

const MASTER_SEED = 20260706   # fixé AVANT toute exécution -- aucun filtrage post-hoc

# vocab et steps recalibrés une seule fois, AVANT de relancer, après que E2 ait montré
# une perte stagnante à ~ln(256) pour LES TROIS bras (y compris `degenerate`, qui n'a
# aucun lien causal avec le greffon) -- signe que le budget (200 pas, vocab=256) ne
# donnait au modèle de base aucune chance d'apprendre la tâche, pas un problème du
# greffon. vocab=50/steps=600 restent fixés APRÈS cet ajustement, quel que soit le
# résultat obtenu.
const CFG = (
    d_model  = 128,
    n_heads  = 4,
    n_blocks = 4,
    vocab    = 50,
    seq_len  = 32,
    k_insert = 2,      # greffe entre bloc 2 et bloc 3
    steps    = 600,    # pas d'entraînement pour E2
    lr       = 1f-3,
    tau_escape = 5,    # nb de pas max attendu pour que |alpha| > 0 (bras rezero)
)

results_dir = joinpath(@__DIR__, "results_surgery")
mkpath(results_dir)
println("Master seed : ", MASTER_SEED, "  (fixé avant toute run -- aucun filtrage post-hoc)")

Master seed : 20260706  (fixé avant toute run -- aucun filtrage post-hoc)


## Section 0 — Adaptateur NeuroDSL

Graphe : `Embedding(token) + Embedding(position) -> LlamaModel -> Linear -> :cross_entropy`
(même câblage que `build_induction_graph`, `src/synthetic_circuits.jl:129-144`, mais avec un
générateur de batch générique au lieu de la tâche d'induction spécifique).

`graft_shadow_block!` (l'adaptateur, ci-dessous) enrichit le `handle` retourné par
`NeuroDSL.graft_shadow_block!` avec une référence au modèle -- nécessaire car le script
original appelle `alpha_grad(handle)`/`branch_grad_norm(handle)` sans passer le modèle
séparément.

In [2]:
mutable struct ExpModel
    g::NeuroDSL.NeuroGraph
    ns::Symbol
    logits_sym::Symbol
    dim::Int
    n_heads::Int
    hidden_dim::Int
    m1::Dict{Symbol,Array{Float32}}
    m2::Dict{Symbol,Array{Float32}}
    t::Ref{Int}
    last_grads::Dict{Symbol,Array{Float32}}   # snapshot pris juste après backward_graph!,
                                               # AVANT adamw_step! -- qui remet le gradient à
                                               # zéro comme dernière étape (même piège que
                                               # capture_snapshot, src/viz.jl). alpha_grad/
                                               # branch_grad_norm lisent CE cache, jamais
                                               # .gradient en direct une fois train_step! fini.
end

"""Construit le modèle Llama-style de référence (N blocs) à partir d'un rng dédié."""
function build_model(cfg::NamedTuple, rng::AbstractRNG)
    dev = NeuroDSL.Backend.CPUDevice()
    ns  = gensym(:expmodel)
    Random.seed!(rand(rng, 1:10^9))   # NeuroDSL initialise ses poids via le RNG global
    g = NeuroDSL.NeuroGraph(namespace=ns, device=dev)
    NeuroDSL.set!(g, :token_ids, ones(Int, cfg.seq_len); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :pos_ids, collect(1:cfg.seq_len); atom_type=NeuroDSL.Datom, namespace=ns)
    tok_emb = NeuroDSL.Embedding(cfg.vocab, cfg.d_model)(g, :token_ids, :tok; namespace=ns)
    pos_emb = NeuroDSL.Embedding(cfg.seq_len, cfg.d_model)(g, :pos_ids, :pos; namespace=ns)
    xsum = :embed_sum
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(xsum, [tok_emb, pos_emb], :add; namespace=ns))
    hidden_dim = 4 * cfg.d_model
    out = NeuroDSL.LlamaModel(cfg.n_blocks, cfg.d_model, cfg.n_heads, hidden_dim)(g, xsum; namespace=ns)
    logits = NeuroDSL.Linear(cfg.d_model, cfg.vocab)(g, out, :lm_head; namespace=ns)
    NeuroDSL.set!(g, :labels, ones(Int, cfg.seq_len); atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [logits, :labels], :cross_entropy; namespace=ns))
    return ExpModel(g, ns, logits, cfg.d_model, cfg.n_heads, hidden_dim,
                     Dict{Symbol,Array{Float32}}(), Dict{Symbol,Array{Float32}}(), Ref(0),
                     Dict{Symbol,Array{Float32}}())
end

"""Forward complet, retourne les logits en Matrix{Float32} (seq_len × vocab)."""
function forward_logits(model::ExpModel, X)::Matrix{Float32}
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Array(NeuroDSL.demand!(model.g, model.logits_sym; namespace=model.ns))
end

"""
Greffe un bloc résiduel gaté entre les blocs k et k+1 (délègue à
NeuroDSL.graft_shadow_block!, src/graph_surgery.jl -- voir sa docstring pour le
détail du mécanisme F(x)=x+alpha*R(x;theta)). Le `handle` est enrichi d'une
référence au modèle pour satisfaire les signatures alpha_grad(handle)/
branch_grad_norm(handle) du script original.
"""
function graft_shadow_block!(model::ExpModel, k::Int; alpha0::Float32,
                             zero_out_proj::Bool, rng::AbstractRNG)
    Random.seed!(rand(rng, 1:10^9))
    after_sym = Symbol(:layer_, k, :_out)
    _, handle = NeuroDSL.graft_shadow_block!(model.g, model.ns, after_sym,
                                              model.dim, model.n_heads, model.hidden_dim;
                                              alpha0=alpha0, zero_out_proj=zero_out_proj)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    for p in NeuroDSL.params(model.g; namespace=model.ns)
        haskey(model.m1, p.name) && continue
        model.m1[p.name] = zeros(Float32, size(p.value)...)
        model.m2[p.name] = zeros(Float32, size(p.value)...)
    end
    return merge(handle, (; model))
end

"""Valeur de la branche R(x) au site de greffe (AVANT multiplication par alpha)."""
function branch_output(handle, model::ExpModel, X)::Array{Float32}
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Array(NeuroDSL.demand!(model.g, handle.R_sym; namespace=model.ns))
end

"""Activations du flux résiduel ENTRANT dans le greffon (pour le scan -0.0)."""
function residual_stream_at(model::ExpModel, k::Int, X)::Array{Float32}
    after_sym = Symbol(:layer_, k, :_out)
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Array(NeuroDSL.demand!(model.g, after_sym; namespace=model.ns))
end

"""Un pas d'entraînement (forward + backward + update AdamW lr) sur (X, Y). Retourne la loss."""
function train_step!(model::ExpModel, X, Y; lr::Float32)::Float32
    model.t[] += 1
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.set!(model.g, :labels, Y; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    loss_val = NeuroDSL.demand!(model.g, :loss; namespace=model.ns)
    NeuroDSL.backward_graph!(model.g, :loss; namespace=model.ns)

    # Snapshot AVANT la mise à jour -- adamw_step! consomme/zéroïfie le gradient vécu
    # comme un effet de bord (voir le champ last_grads ci-dessus).
    empty!(model.last_grads)
    for p in NeuroDSL.params(model.g; namespace=model.ns)
        p.gradient === nothing && continue
        model.last_grads[p.name] = copy(Array(p.gradient))
    end

    for p in NeuroDSL.params(model.g; namespace=model.ns)
        p.gradient === nothing && continue
        if !haskey(model.m1, p.name)
            model.m1[p.name] = zeros(Float32, size(p.value)...)
            model.m2[p.name] = zeros(Float32, size(p.value)...)
        end
        NeuroDSL.adamw_step!(NeuroDSL.Backend.CPUDevice(), p.value, p.gradient,
                              model.m1[p.name], model.m2[p.name],
                              lr, 0.9f0, 0.999f0, 1f-8, model.t[], 1f0, 0f0)
    end
    loss_scalar = Float32(sum(Array(loss_val)))
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return loss_scalar
end

"""Norme L2 du gradient des paramètres theta du greffon (APRÈS backward, AVANT update)."""
function branch_grad_norm(handle)::Float64
    model = handle.model
    mha = Symbol(handle.prefix, :_mha)
    theta_syms = [Symbol(mha,:_q_W), Symbol(mha,:_k_W), Symbol(mha,:_v_W), Symbol(mha,:_output_W),
                  Symbol(handle.prefix,:_mlp_w1), Symbol(handle.prefix,:_mlp_w2), Symbol(handle.prefix,:_mlp_w3)]
    total = 0.0
    for sym in theta_syms
        gr = get(model.last_grads, sym, nothing)
        gr === nothing && continue
        total += sum(abs2, gr)
    end
    return sqrt(total)
end

"""Valeur courante du gate alpha, et son gradient au dernier backward."""
alpha_value(handle)::Float32 = Array(NeuroDSL.node(handle.model.g, handle.alpha_sym; namespace=handle.model.ns).value)[1]
function alpha_grad(handle)::Float32
    gr = get(handle.model.last_grads, handle.alpha_sym, nothing)
    return gr === nothing ? 0f0 : gr[1]
end

"""Loss d'évaluation (sans update) sur (X, Y)."""
function eval_loss(model::ExpModel, X, Y)::Float32
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.set!(model.g, :labels, Y; atom_type=NeuroDSL.Datom, namespace=model.ns)
    NeuroDSL.invalidate_all!(model.g; namespace=model.ns)
    return Float32(sum(Array(NeuroDSL.demand!(model.g, :loss; namespace=model.ns))))
end

"""
Générateur de batch : tâche d'induction (Olsson et al. 2022), pas une copie
décalée i.i.d. -- pour un modèle CAUSAL, `Y[i]=X[i+1]` avec `X` i.i.d. est
information-théoriquement inapprenable (Y[i] est indépendant de tout ce qui
est visible en position i). La tâche d'induction (préfixe répété deux fois)
est la version la plus proche qui reste réellement apprenable, déjà prouvée
dans ce dépôt (`sample_induction_sequence`, `src/synthetic_circuits.jl`).
"""
function make_batch(rng::AbstractRNG, cfg)
    return NeuroDSL.sample_induction_sequence(rng, cfg.vocab, cfg.seq_len ÷ 2)
end

println("Section 0 (adaptateur) chargée -- 8 fonctions reliées à la vraie API NeuroDSL.")

Section 0 (adaptateur) chargée -- 8 fonctions reliées à la vraie API NeuroDSL.


## Section 2 — E1 : bit-exactness

Reprend tel quel le protocole du script fourni : logits avant greffe, greffe
Gradient Shadowing (`alpha=0`, theta aléatoire), logits après greffe (aucun pas
d'entraînement), comparaison bit-à-bit (`reinterpret(UInt32, ...)`).

## Section 4 — E3 : coût de la chirurgie vs profondeur d'insertion (localité)

Claim testée : le coût de la chirurgie (greffe+invalidation, puis première
recomputation) est proportionnel à la taille du cône aval `|V_k+|`, PAS à la
taille totale du graphe -- il doit donc décroître avec la profondeur `k`.

**Correction faite en reliant l'adaptateur** (pas juste "brancher 2 fonctions
manquantes") : coller tel quel le script `experiments_surgery_E3.jl` fourni
**redéfinissait** `build_model`/`forward_logits`/`graft_shadow_block!` avec ses
propres stubs `error(...)`, écrasant les vraies implémentations déjà chargées
en Section 0 -- d'où l'erreur "ADAPTER non lié" malgré leur présence plus haut
dans ce même notebook. Au-delà de ce renommage, les versions E1/E2 de
`forward_logits`/`graft_shadow_block!` appellent `invalidate_all!` (nécessaire
pour l'entraînement, où `X` change à chaque pas) -- les réutiliser telles
quelles ici invaliderait le graphe ENTIER à chaque mesure et fausserait
exactement ce qu'E3 cherche à prouver (coût ∝ cône, pas taille totale). E3
utilise donc des variantes dédiées, en lecture seule, qui laissent
l'invalidation ciblée déjà intégrée à `addrule!`/`graft_shadow_block!` faire
son travail sans la court-circuiter.

In [3]:
"Scan des préconditions de la Proposition 2. Retourne (ok, rapport)."
function check_preconditions(model, handle, X, k)
    res = residual_stream_at(model, k, X)
    r   = branch_output(handle, model, X)

    n_negzero = count(v -> iszero(v) && signbit(v), res)
    n_nonfin  = count(!isfinite, r)

    ok  = (n_negzero == 0) && (n_nonfin == 0)
    rep = @sprintf("""
    Préconditions (Prop. 2) :
      composantes -0.0 dans le flux résiduel : %d / %d   %s
      valeurs non finies dans R(x)           : %d / %d   %s
    """, n_negzero, length(res), n_negzero == 0 ? "[OK]" : "[VIOLATION -> exactitude fonctionnelle seulement]",
         n_nonfin, length(r),   n_nonfin == 0 ? "[OK]" : "[VIOLATION -> NaN attendu, greffe NON exacte]")
    return ok, rep
end

function run_E1(master_seed)
    println("="^70); println("E1 — BIT-EXACTNESS  (Théorème 1 + Proposition 2)"); println("="^70)

    rng_model = MersenneTwister(hash((master_seed, :model)))
    rng_data  = MersenneTwister(hash((master_seed, :data)))
    rng_graft = MersenneTwister(hash((master_seed, :graft)))

    model = build_model(CFG, rng_model)
    X, _  = make_batch(rng_data, CFG)

    y_before = forward_logits(model, X)
    bits_before = reinterpret(UInt32, vec(y_before))

    handle = graft_shadow_block!(model, CFG.k_insert;
                                 alpha0=0f0, zero_out_proj=false, rng=rng_graft)

    pre_ok, pre_report = check_preconditions(model, handle, X, CFG.k_insert)
    print(pre_report)

    y_after = forward_logits(model, X)
    bits_after = reinterpret(UInt32, vec(y_after))

    n_mismatch = count(bits_before .!= bits_after)
    n_total    = length(bits_before)

    verdict = n_mismatch == 0 ? "BIT-EXACT [OK]" :
              (y_before ≈ y_after ? "fonctionnellement égal mais PAS bit-exact [X]" :
                                    "NON EXACT [XX] (bug de greffe probable)")

    report = @sprintf("""
    %s
    Éléments comparés : %d
    Mismatches bit    : %d  (%.2e %%)
    Max |Δ| flottant  : %.3e
    Préconditions     : %s
    Verdict           : %s
    """, "-"^50, n_total, n_mismatch, 100n_mismatch/n_total,
         maximum(abs.(y_before .- y_after)),
         pre_ok ? "satisfaites" : "VIOLÉES (voir ci-dessus)", verdict)

    println(report)
    if n_mismatch > 0
        idx = findall(bits_before .!= bits_after)[1:min(5, n_mismatch)]
        println("Premiers écarts (index, avant, après) :")
        for i in idx
            @printf("  [%d]  %a  ->  %a\n", i, vec(y_before)[i], vec(y_after)[i])
        end
    end
    write(joinpath(results_dir, "E1_report.txt"), pre_report * report)
    return n_mismatch == 0
end

run_E1 (generic function with 1 method)

In [4]:
ok1 = run_E1(MASTER_SEED)

E1 — BIT-EXACTNESS  (Théorème 1 + Proposition 2)
✅ Op :scalar_gate registered
Préconditions (Prop. 2) :
  composantes -0.0 dans le flux résiduel : 0 / 4096   [OK]
  valeurs non finies dans R(x)           : 0 / 4096   [OK]
--------------------------------------------------
Éléments comparés : 1600
Mismatches bit    : 0  (0.00e+00 %)
Max |Δ| flottant  : 0.000e+00
Préconditions     : satisfaites
Verdict           : BIT-EXACT [OK]



true

## Section 3 — E2 : plasticité vs point de selle (3 bras, master seed partagé)

In [5]:
const ARMS = (
    (name="rezero",     alpha0=0f0, zero_out=false),  # Gradient Shadowing (Prop. 4)
    (name="net2net",    alpha0=1f0, zero_out=true),   # Fixup/Net2Net
    (name="degenerate", alpha0=0f0, zero_out=true),   # Prop. 3 : doit rester gelé
)

function run_arm(arm, master_seed)
    rng_model = MersenneTwister(hash((master_seed, :model)))
    rng_data  = MersenneTwister(hash((master_seed, :data)))
    rng_graft = MersenneTwister(hash((master_seed, :graft)))

    model  = build_model(CFG, rng_model)
    handle = graft_shadow_block!(model, CFG.k_insert;
                                 alpha0=arm.alpha0, zero_out_proj=arm.zero_out,
                                 rng=rng_graft)

    hist = zeros(Float64, CFG.steps, 5)  # step | loss | |alpha| | grad_alpha | grad_theta

    for t in 1:CFG.steps
        X, Y = make_batch(rng_data, CFG)
        loss = train_step!(model, X, Y; lr=CFG.lr)
        hist[t, :] .= (t, loss, abs(alpha_value(handle)),
                       abs(alpha_grad(handle)), branch_grad_norm(handle))
    end

    path = joinpath(results_dir, "E2_$(arm.name).csv")
    open(path, "w") do io
        println(io, "step,loss,abs_alpha,abs_grad_alpha,grad_theta_norm")
        writedlm(io, hist, ',')
    end
    return hist
end

function run_E2(master_seed)
    println("\n"); println("="^70)
    println("E2 — PLASTICITÉ / POINT SELLE  (Propositions 3 et 4)"); println("="^70)

    results = Dict{String, Matrix{Float64}}()
    for arm in ARMS
        println("  bras `$(arm.name)`  (alpha0=$(arm.alpha0), zero_out_proj=$(arm.zero_out)) ...")
        results[arm.name] = run_arm(arm, master_seed)
    end

    io = IOBuffer()
    println(io, "-"^70)
    @printf(io, "%-12s | %10s | %10s | %12s | %12s\n",
            "bras", "loss t=1", "loss fin", "|alpha| fin", "‖∇θ‖ max")
    println(io, "-"^70)
    for arm in ARMS
        h = results[arm.name]
        @printf(io, "%-12s | %10.4f | %10.4f | %12.3e | %12.3e\n",
                arm.name, h[1,2], h[end,2], h[end,3], maximum(h[:,5]))
    end
    println(io, "-"^70)

    h_rz, h_dg = results["rezero"], results["degenerate"]
    checks = [
        ("Prop.4 — dL/dalpha != 0 au premier backward (rezero)",
            h_rz[1,4] > 0),
        ("Prop.4 — |alpha| > 0 en <= $(CFG.tau_escape) pas (rezero)",
            any(h_rz[1:CFG.tau_escape, 3] .> 0)),
        ("Prop.4 — ‖∇θ‖ devient non nul après l'escape (rezero)",
            maximum(h_rz[:,5]) > 0),
        ("Prop.3 — dL/dalpha == 0 sur toute la run (degenerate)",
            all(h_dg[:,4] .== 0)),
        ("Prop.3 — ‖∇θ‖ == 0 sur toute la run (degenerate)",
            all(h_dg[:,5] .== 0)),
        ("Prop.3 — alpha ne bouge jamais (degenerate)",
            all(h_dg[:,3] .== 0)),
        ("Net2Net — le bras (b) apprend (loss décroît)",
            results["net2net"][end,2] < results["net2net"][1,2]),
    ]
    println(io, "\nAssertions :")
    all_ok = true
    for (label, ok) in checks
        all_ok &= ok
        println(io, (ok ? "  [PASS] " : "  [FAIL] ") * label)
    end
    summary = String(take!(io))
    println(summary)
    write(joinpath(results_dir, "E2_summary.txt"), summary)
    return all_ok
end

run_E2 (generic function with 1 method)

In [6]:
ok2 = run_E2(MASTER_SEED)



E2 — PLASTICITÉ / POINT SELLE  (Propositions 3 et 4)
  bras `rezero`  (alpha0=0.0, zero_out_proj=false) ...
  bras `net2net`  (alpha0=1.0, zero_out_proj=true) ...
  bras `degenerate`  (alpha0=0.0, zero_out_proj=true) ...
----------------------------------------------------------------------
bras         |   loss t=1 |   loss fin |  |alpha| fin |     ‖∇θ‖ max
----------------------------------------------------------------------
rezero       |     3.9236 |     1.8771 |    2.676e-04 |    3.277e-02
net2net      |     3.9236 |     1.9517 |    7.410e-01 |    1.649e+00
degenerate   |     3.9236 |     1.9252 |    0.000e+00 |    0.000e+00
----------------------------------------------------------------------

Assertions :
  [PASS] Prop.4 — dL/dalpha != 0 au premier backward (rezero)
  [PASS] Prop.4 — |alpha| > 0 en <= 5 pas (rezero)
  [PASS] Prop.4 — ‖∇θ‖ devient non nul après l'escape (rezero)
  [PASS] Prop.3 — dL/dalpha == 0 sur toute la run (degenerate)
  [PASS] Prop.3 — ‖∇θ‖ == 0 sur tou

true

In [7]:
println("="^70)
println(ok1 && ok2 ? ">>> TOUTES LES CLAIMS VALIDÉES -- résultats dans results_surgery/."
                   : ">>> AU MOINS UN ÉCHEC -- voir results_surgery/ ; ne PAS publier avant résolution.")

>>> TOUTES LES CLAIMS VALIDÉES -- résultats dans results_surgery/.


In [8]:
# Adaptateur E3 -- N'ÉCRASE PAS build_model/forward_logits/graft_shadow_block! de
# la Section 0 : n'ajoute que ce qui manque (downstream_cone_size, graph_size)
# plus des variantes EN LECTURE SEULE nécessaires pour ne pas fausser la mesure.

"""Taille du cône aval de layer_k_out, en LECTURE SEULE (aucune mutation du
graphe) -- réutilise directement _downstream_nodes (src/patching.jl), la même
traversée que restore_from_cache!/sweep_patch_sites!.

IMPORTANT (bug trouvé et corrigé en écrivant ce notebook) : _downstream_nodes
ne continue sa traversée à travers un consommateur que s'il est ENCORE valide
(`out_nd.valid`, src/patching.jl:136) -- c'est ce qui la rend correcte à
utiliser juste avant un patch (elle s'arrête là où une invalidation précédente
s'est déjà arrêtée). Appelée sur un graphe qui vient d'être CONSTRUIT (jamais
`demand!`), tout est encore `valid=false` par défaut et elle s'arrête après un
seul saut -- vérifié empiriquement (3 nœuds au lieu de 288 sur un LlamaModel
à 8 couches, comparé à une BFS de vérité terrain indépendante sur g.rules).
Precondition documentée dans src/patching.jl:113 : le graphe doit être
entièrement valide (un demand! complet) avant d'appeler cette fonction."""
function downstream_cone_size(model::ExpModel, k::Int)::Int
    after_sym = Symbol(:layer_, k, :_out)
    return length(NeuroDSL._downstream_nodes(model.g, after_sym, model.ns))
end

"""Nombre total de nœuds du graphe (pour rapporter cône / total)."""
graph_size(model::ExpModel)::Int = length(model.g.nodes[model.ns])

"""Fixe :token_ids UNE fois (invalidation ciblée normale de set!) -- à utiliser
avant le tout premier forward de measure_one, pour amener le graphe à un état
entièrement valide avant de chronométrer quoi que ce soit."""
function set_input!(model::ExpModel, X)
    NeuroDSL.set!(model.g, :token_ids, X; atom_type=NeuroDSL.Datom, namespace=model.ns)
end

"""Forward SANS invalidation forcée -- laisse demand! ne recalculer que ce qui
est réellement invalide, exactement le point que E3 mesure. À n'utiliser que
lorsque :token_ids est DÉJÀ à jour sur le graphe (X inchangé depuis set_input!)."""
function forward_logits_readonly(model::ExpModel)
    return Array(NeuroDSL.demand!(model.g, model.logits_sym; namespace=model.ns))
end

"""Greffe SANS le invalidate_all!/bookkeeping AdamW de la version E1/E2 -- E3 ne
s'entraîne jamais, et invalidate_all! aurait invalidé tout le graphe au lieu du
seul cône aval, ce qui aurait ruiné la mesure elle-même."""
function graft_shadow_block_readonly!(model::ExpModel, k::Int; alpha0::Float32,
                                       zero_out_proj::Bool, rng::AbstractRNG)
    Random.seed!(rand(rng, 1:10^9))
    after_sym = Symbol(:layer_, k, :_out)
    _, handle = NeuroDSL.graft_shadow_block!(model.g, model.ns, after_sym,
                                              model.dim, model.n_heads, model.hidden_dim;
                                              alpha0=alpha0, zero_out_proj=zero_out_proj)
    return merge(handle, (; model))
end

# ── Configuration E3 (distincte de CFG : modèle plus profond pour avoir
#    plusieurs k à comparer -- pas de batch, comme le reste du notebook) ──────
const CFG3 = (
    d_model  = 128,
    n_heads  = 4,
    n_blocks = 8,          # plus profond que E1/E2 : il faut plusieurs k distincts
    vocab    = 256,
    seq_len  = 32,
    n_reps   = 15,         # répétitions par profondeur (médiane tronquée)
    trim     = 0.10,       # fraction tronquée de chaque côté
    warmup_reps = 3,       # greffes de chauffe (compilation) non mesurées
)
const DEPTHS = collect(1:(CFG3.n_blocks - 1))   # insertion après le bloc k

trimmed_median(v, trim) = begin
    s = sort(v); n = length(s); cut = floor(Int, trim * n)
    median(s[(cut + 1):(n - cut)])
end

"Entrée déterministe (une séquence, pas de dimension batch)."
make_input(rng, cfg) = rand(rng, 1:cfg.vocab, cfg.seq_len)

"""
Un échantillon de mesure à profondeur k :
  1. modèle FRAIS (la greffe mute le graphe -> jamais réutilisé entre samples)
  2. forward initial complet (le graphe est entièrement valide avant la greffe,
     sinon la 'recomputation' mesurerait aussi du travail préexistant)
  3. chronométrage (a) : greffe + invalidation CIBLÉE [graft_shadow_block_readonly!]
  4. chronométrage (b) : première recomputation, X inchangé [forward_logits_readonly]
Retourne (t_graft_ms, t_recompute_ms, allocs_graft_bytes).
"""
function measure_one(k::Int, rep::Int, master_seed)
    rng_model = MersenneTwister(hash((master_seed, :model, k, rep)))
    rng_data  = MersenneTwister(hash((master_seed, :data,  k, rep)))
    rng_graft = MersenneTwister(hash((master_seed, :graft, k, rep)))

    model = build_model(CFG3, rng_model)
    X = make_input(rng_data, CFG3)
    set_input!(model, X)
    forward_logits_readonly(model)                 # graphe entièrement valide

    stats = @timed graft_shadow_block_readonly!(model, k;
                alpha0=0f0, zero_out_proj=false, rng=rng_graft)
    t_graft_ms   = stats.time * 1000
    allocs_graft = stats.bytes

    t0 = time_ns()
    forward_logits_readonly(model)                 # recompute -- X inchangé, cône seul invalide
    t_recompute_ms = (time_ns() - t0) / 1e6

    return t_graft_ms, t_recompute_ms, allocs_graft
end

function run_E3(master_seed)
    println("="^70)
    println("E3 — COÛT DE LA CHIRURGIE vs PROFONDEUR  (Théorème de localité)")
    println("="^70)
    println("Master seed : $master_seed   |   profondeurs : $(DEPTHS)")

    # Cônes structurels : le graphe DOIT être entièrement valide (demand! complet)
    # avant d'appeler downstream_cone_size -- voir sa docstring pour le bug que
    # ça a fait apparaître la première fois (mesure sur un graphe jamais calculé).
    rng0  = MersenneTwister(hash((master_seed, :model, 0, 0)))
    rng0d = MersenneTwister(hash((master_seed, :data,  0, 0)))
    m0    = build_model(CFG3, rng0)
    set_input!(m0, make_input(rng0d, CFG3))
    forward_logits_readonly(m0)
    total = graph_size(m0)
    cones = Dict(k => downstream_cone_size(m0, k) for k in DEPTHS)
    println("\nGraphe : $total nœuds. Cônes aval par profondeur :")
    for k in DEPTHS
        @printf("  k=%d : %d nœuds (%.1f%%)\n", k, cones[k], 100cones[k]/total)
    end

    println("\nWarm-up ($(CFG3.warmup_reps) greffes jetables)...")
    for w in 1:CFG3.warmup_reps
        measure_one(first(DEPTHS), -w, master_seed)
        measure_one(last(DEPTHS),  -w, master_seed)
    end

    graft_t   = Dict(k => Float64[] for k in DEPTHS)
    recomp_t  = Dict(k => Float64[] for k in DEPTHS)
    graft_b   = Dict(k => Float64[] for k in DEPTHS)
    for rep in 1:CFG3.n_reps
        for k in DEPTHS
            tg, tr, ab = measure_one(k, rep, master_seed)
            push!(graft_t[k], tg); push!(recomp_t[k], tr); push!(graft_b[k], ab)
        end
        rep % 5 == 0 && println("  ... répétition $rep/$(CFG3.n_reps)")
    end

    rows = zeros(Float64, length(DEPTHS), 6)
    for (i, k) in enumerate(DEPTHS)
        rows[i, :] .= (k, cones[k],
                       trimmed_median(graft_t[k],  CFG3.trim),
                       trimmed_median(recomp_t[k], CFG3.trim),
                       trimmed_median(graft_t[k],  CFG3.trim) +
                       trimmed_median(recomp_t[k], CFG3.trim),
                       trimmed_median(graft_b[k],  CFG3.trim) / 1024)  # KiB
    end
    open(joinpath(results_dir, "E3_cost_vs_depth.csv"), "w") do io
        println(io, "depth_k,cone_size,graft_ms,recompute_ms,total_ms,graft_allocs_KiB")
        writedlm(io, rows, ',')
    end

    io = IOBuffer()
    println(io, "-"^70)
    @printf(io, "%4s | %9s | %10s | %13s | %10s\n",
            "k", "cône", "greffe(ms)", "recompute(ms)", "total(ms)")
    println(io, "-"^70)
    for i in 1:size(rows, 1)
        @printf(io, "%4d | %9d | %10.3f | %13.3f | %10.3f\n",
                Int(rows[i,1]), Int(rows[i,2]), rows[i,3], rows[i,4], rows[i,5])
    end
    println(io, "-"^70)

    recomps = rows[:, 4]
    csizes  = rows[:, 2]
    r = cor(csizes, recomps)
    mono_ok = all(recomps[i] >= recomps[i+1] * 0.90 for i in 1:length(recomps)-1)

    checks = [
        ("Localité — coût de recompute décroît (monotone, tol. bruit 10%) avec k", mono_ok),
        ("Localité — corrélation recompute vs taille de cône r > 0.95 (r = $(round(r, digits=4)))", r > 0.95),
        ("Localité — coût k=$(last(DEPTHS)) < 50% du coût k=$(first(DEPTHS))",
            recomps[end] < 0.5 * recomps[1]),
        ("Greffe+invalidation « petite » : médiane < coût de recompute à k=1",
            rows[argmax(rows[:,3]), 3] < recomps[1]),
    ]
    println(io, "\nAssertions :")
    all_ok = true
    for (label, ok) in checks
        all_ok &= ok
        println(io, (ok ? "  [PASS] " : "  [FAIL] ") * label)
    end
    println(io, """

    Lecture pour le papier :
      - la colonne recompute(ms) vs cone_size est la validation directe du
        Théorème de localité (coût ∝ |V_k+|, indépendant de la taille totale) ;
      - graft(ms) mesure la partie bookkeeping (construction + invalidation) --
        si elle NE décroît PAS avec k alors qu'elle devrait suivre le cône,
        suspecter un scan O(|cone|×|rules|) plutôt que l'index de consommateurs.
    """)
    summary = String(take!(io))
    println(summary)
    write(joinpath(results_dir, "E3_summary.txt"), summary)
    return all_ok
end

run_E3 (generic function with 1 method)

In [9]:
ok3 = run_E3(MASTER_SEED)

E3 — COÛT DE LA CHIRURGIE vs PROFONDEUR  (Théorème de localité)
Master seed : 20260706   |   profondeurs : [1, 2, 3, 4, 5, 6, 7]

Graphe : 412 nœuds. Cônes aval par profondeur :
  k=1 : 289 nœuds (70.1%)
  k=2 : 248 nœuds (60.2%)
  k=3 : 207 nœuds (50.2%)
  k=4 : 166 nœuds (40.3%)
  k=5 : 125 nœuds (30.3%)
  k=6 : 84 nœuds (20.4%)
  k=7 : 43 nœuds (10.4%)

Warm-up (3 greffes jetables)...
  ... répétition 5/15
  ... répétition 10/15
  ... répétition 15/15
----------------------------------------------------------------------
   k |      cône | greffe(ms) | recompute(ms) |  total(ms)
----------------------------------------------------------------------
   1 |       289 |      0.784 |        15.488 |     16.272
   2 |       248 |      1.032 |        13.868 |     14.900
   3 |       207 |      0.783 |        11.531 |     12.315
   4 |       166 |      0.722 |         9.785 |     10.507
   5 |       125 |      0.745 |         7.808 |      8.553
   6 |        84 |      0.819 |         6.273

true